In [ ]:
from rtmpose_processor import RTMPoseProcessor
from exercise_counters import ExerciseCounter
# 只使用CPU设备
device = 'cpu'

# 设置默认模型模式
model_mode = 'lightweight'

# 创建运动计数器实例
exercise_counter = ExerciseCounter()
pose_processor = RTMPoseProcessor(
            exercise_counter=exercise_counter,
            mode=model_mode,
            backend='onnxruntime',
            device=device
        )

'''
squat 深蹲
pushup 俯卧撑
situp 仰卧起坐
bicep_curl 二头肌弯举
lateral_raise	侧平举
overhead_press	头顶推举
leg_raise	抬腿
knee_raise	提膝
left_knee_press	左膝下压
right_knee_press	右膝下压
'''
action_dic={'squat':'深蹲',
            'pushup':'俯卧撑',
            'situp':'仰卧起坐',
            'bicep_curl':'二头肌弯举',
            'lateral_raise':'侧平举',
            'overhead_press':'头顶推举',
            'leg_raise':'抬腿',
            'knee_raise':'提膝',
            'left_knee_press':'左膝下压',
            'right_knee_press':'右膝下压'
            }
action_list = ['squat','pushup','situp','bicep_curl','lateral_raise','overhead_press','leg_raise','knee_raise','left_knee_press','right_knee_press']


In [ ]:
from PIL import Image, ImageDraw, ImageFont
from key import Button,language
import xgoscreen.LCD_2inch as LCD_2inch
la=language()
button = Button()
splash_theme_color = (0,0,0)
mydisplay = LCD_2inch.LCD_2inch()
mydisplay.Init()
mydisplay.clear()
# Init Splash
splash = Image.new("RGB", (mydisplay.height, mydisplay.width), splash_theme_color)
draw = ImageDraw.Draw(splash)
mydisplay.ShowImage(splash)


In [ ]:
import ipywidgets.widgets as widgets
from IPython.display import display
import cv2
#显示摄像头组件 Display camera components
image_widget = widgets.Image(format='jpeg', width=320, height=240)

#bgr8转jpeg格式  bgr8 to jpeg format
import enum
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

In [ ]:

def lcd_draw_string(
    draw,  # 修改为直接使用draw对象 Modify to directly use the draw object
    x,
    y,
    text,
    color=(255, 255, 255),
    font_size=16,
    max_width=340,
    max_lines=5,
    clear_area=False
):
    font = ImageFont.truetype("/home/pi/model/msyh.ttc", font_size)
    line_height = font_size + 2
    total_height = max_lines * line_height
    
    if clear_area:
        draw.rectangle((x, y, x + max_width, y + total_height), fill=(15, 21, 46))
    
    lines = []
    current_line = ""
    
    # 处理文本中的换行符 Processing line breaks in text
    paragraphs = text.split('\n')
    
    for para in paragraphs:
        words = []
        # 将中英文组合拆分为可分割的单元 Split the combination of Chinese and English into separable units
        temp_word = ""
        for char in para:
            # 判断是否为ASCII字符(英文) Determine whether it is an ASCII character (in English)
            if ord(char) < 256:
                if temp_word and not temp_word.isascii():
                    words.append(temp_word)
                    temp_word = ""
                temp_word += char
            else:
                if temp_word and temp_word.isascii():
                    words.append(temp_word)
                    temp_word = ""
                temp_word += char
        if temp_word:
            words.append(temp_word)
        
        current_line = ""
        for word in words:
            # 测试添加这个词是否会超出宽度 Test whether adding this word will exceed the width
            test_line = current_line + word
            if font.getlength(test_line) <= max_width:
                current_line = test_line
            else:
                if current_line:  # 如果当前行有内容，先保存 If there is content in the current line, save it first
                    lines.append(current_line)
                # 处理一个词就超长的情况 Dealing with situations where one word is too long
                if font.getlength(word) > max_width:
                    # 对超长英文单词进行分割 Segmentation of Extra Long English Words
                    if word.isascii():
                        split_pos = 0
                        while split_pos < len(word):
                            remaining = len(word) - split_pos
                            # 找最大能显示的长度 Find the maximum length that can be displayed
                            for l in range(remaining, 0, -1):
                                if font.getlength(word[split_pos:split_pos+l]) <= max_width:
                                    lines.append(word[split_pos:split_pos+l])
                                    split_pos += l
                                    break
                    else:  
                        for char in word:
                            lines.append(char)
                    current_line = ""
                else:
                    current_line = word
        if current_line:
            lines.append(current_line)
    
    # 限制最大行数 Limit the maximum number of rows
    if max_lines:
        lines = lines[:max_lines]
    
    # 绘制文本 Drawing Text
    for i, line in enumerate(lines):
        draw.text((x, y + i * line_height), line, fill=color, font=font)
   

In [ ]:
display(image_widget)  
i = 0   
while True:

    exercise_type = action_list[i]
    splash = Image.new("RGB", (mydisplay.height, mydisplay.width), splash_theme_color)
    draw = ImageDraw.Draw(splash)
    #Show Wake Up Call
    if la=="cn":
        text1=f"当前检测的运动为\n{action_dic[exercise_type]}({exercise_type})"
    else:
        text1=f"The current detected motion is\n{exercise_type}"
              
    lcd_draw_string(
        draw,
        x=50,
        y=60,
        text=text1,
        color=(255, 255, 255),
        font_size=26,
        max_width=240,
        max_lines=5,
        clear_area=False
    )
    mydisplay.ShowImage(splash)
    if button.press_d():
        i += 1
        if i >= len(action_list):  
            i = 0
        exercise_type = action_list[i]
    if button.press_c():
        break
    if button.press_b():
        #exit()
        import os
        os._exit(1)
   

from key import Button,language
import cv2
button = Button()
la=language()
from picamera2 import Picamera2
picam2 = Picamera2()
picam2.configure(picam2.create_preview_configuration(main={"format": 'RGB888', "size": (320, 240)}))
picam2.start()

font = cv2.FONT_HERSHEY_SIMPLEX

# 改为使用PIL绘制中文，再转为OpenCV格式： Change to using PIL to draw Chinese, and then convert to OpenCV format:
from PIL import ImageFont, ImageDraw, Image
import numpy as np

# 在循环前定义中文字体 Define Chinese font before loop
font_path = "/home/pi/model/msyh.ttc"  
font_size = 20
pil_font = ImageFont.truetype(font_path, font_size)

while True:
    frame = picam2.capture_array()
    processed_frame, current_angle, keypoints = pose_processor.process_frame(frame, exercise_type)
    
    # 将OpenCV图像转为PIL图像 # Convert OpenCV images to PIL images
    pil_img = Image.fromarray(cv2.cvtColor(processed_frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil_img)
    
    # 组合显示文本 Combine display text
    if la == "cn":
        display_text = f"{action_dic[exercise_type]}({exercise_type})"  # 中文显示  Chinese display
    else:
        display_text = f"{exercise_type}"  # 英文显示 English display
        
    current_count = exercise_counter.counter
    
    # 计算文本位置 Calculate text position
    text_width = draw.textlength(display_text, font=pil_font)
    count_width = draw.textlength(str(current_count), font=pil_font)
    right_margin = 5
    text_x = processed_frame.shape[1] - text_width - right_margin
    count_x = processed_frame.shape[1] - count_width - right_margin
    
    # 绘制文本 Drawing Text
    draw.text((text_x, 30), display_text, font=pil_font, fill=(0, 255, 0))
    draw.text((count_x, 60), str(current_count), font=pil_font, fill=(0, 255, 0))
    
    # 转换回OpenCV格式 Convert back to OpenCV format
    processed_frame = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    
    r, g, b = cv2.split(processed_frame)
    myimg = cv2.merge((b, g, r))
    
    image_widget.value = bgr8_to_jpeg(myimg)
    
    # 显示图像 display image
    imgok = Image.fromarray(processed_frame)
    mydisplay.ShowImage(imgok)
    
    if button.press_b():
        break
